In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers,models
import shutil as shut
import zipfile as zip
import os
import tensorflow as tf
from PIL import Image
import numpy as np

In [2]:
#@title startup
with zip.ZipFile("dataset2.zip","r") as zip_ref:
  zip_ref.extractall('dataset2_unzipped')
  print("succesfully unzipped dataset2")

# deletes unnesery files
src = "dataset2_unzipped/dataset2"
dst = "dataset2_unzipped"
for folder in ["lemon","lime"]:
  shut.move(os.path.join(src,folder),os.path.join(dst,folder))
shut.rmtree(src,ignore_errors = True)
print(os.listdir("dataset2_unzipped"))

succesfully unzipped dataset2
['lemon', 'lime']


In [19]:
#@title compiler
datagen = ImageDataGenerator(rescale = 1./255)

train = datagen.flow_from_directory(
    'dataset2_unzipped',
    target_size = (150,150),
    # resize to 150 by 150 pixels
    batch_size = 1 ,#number of images per batch
    class_mode = 'binary', #Two Classes
    shuffle = True #shuffle the images
  )


Found 11 images belonging to 2 classes.


In [20]:
model = models.Sequential([
    layers.Conv2D(8,(3,3),activation = 'relu',input_shape = (150,150,3)),
    layers.MaxPool2D(2,2),
    layers.Conv2D(16,(3,3),activation = 'relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(32,activation='relu'),
    layers.Dense(1,activation='sigmoid')
])
model.compile(optimizer = 'adam',loss = 'binary_crossentropy',metrics = ['accuracy'])

In [22]:
model.fit(train,epochs=5)

Epoch 1/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 1.0000 - loss: 0.2446
Epoch 2/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 1.0000 - loss: 0.1950
Epoch 3/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 1.0000 - loss: 0.1309
Epoch 4/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 1.0000 - loss: 0.1223
Epoch 5/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 1.0000 - loss: 0.0790


In [23]:
model.save('fruit_classifier.keras')

In [26]:
model = tf.keras.models.load_model('fruit_classifier.keras')
test_img = Image.open('test.jpg').resize((150,150))
x = np.expand_dims(np.array(test_img)/255.0,axis=0)
pred = model.predict(x)

print(train.class_indices)
print(pred[0][0])
if (pred[0][0]>0.5) :
  print("lime")
else :
  print('lemon')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
{'lemon': 0, 'lime': 1}
0.9422702
lime
